In [3]:
import numpy as np
import matplotlib.pyplot as plt
import random 
from sklearn.manifold import TSNE

In [ ]:
## Configuration and File Paths
train_labels_fpath = "/mnist-dataset/train-labels-idx1-ubyte/train-labels-idx1-ubyte"
train_images_fpath = "/mnist-dataset/train-images-idx3-ubyte/train-images-idx3-ubyte"
test_labels_fpath = "/mnist-dataset/t10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte"
test_images_fpath = "/mnist-dataset/t10k-images-idx3-ubyte/t10k-images-idx3-ubyte"

random.seed(2023115)

In [ ]:
def load_dataset(image_file, label_file):
    with open(label_file, 'rb') as f:
        magic, num = np.frombuffer(f.read(8), dtype=np.uint32)
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    with open(image_file, 'rb') as f:
        magic, num, rows, cols = np.frombuffer(f.read(16), dtype=np.uint32)
        images = np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows, cols)
    return images, labels
    

In [ ]:
def preprocess(images):
    max_pixel_value = 255.0
    samples_cnt = images.shape[0]
    flatten_images = images.reshape(samples_cnt, -1)
    norm_images = flatten_images / max_pixel_value
    return norm_images

In [ ]:
def sample_data(images, labels, sample_size = 100):
    classes = np.unique(labels)
    sampled_images = []
    sampled_labels = []
    for cls in classes:
        cls_indices = np.where(labels == cls)[0]
        sampled_indices = np.random.choice(cls_indices, sample_size, replace=False)
        sampled_images.append(images[sampled_indices])
        sampled_labels.append(labels[sampled_indices])

    X = np.vstack(sampled_images)
    y = np.hstack(sampled_labels)

    permuted_indices = np.random.permutation(len(y))
    return X[permuted_indices], y[permuted_indices]

In [ ]:
def filter012(images, labels):
    classes=(0, 1, 2)
    indices = np.where((labels == classes[0]) | (labels == classes[1]) | (labels == classes[2]))[0]
    return images[indices], labels[indices]

In [ ]:
def MLEstimate(train_images, train_labels):
    reg = 1e-5
    classes = np.unique(train_labels)
    params={}
    for c in classes:
        class_images = train_images[train_labels == c]
        N = class_images.shape[0]
        mean = np.mean(class_images, axis=0)       ## u
        centered_images = class_images - mean ## (x-u)
        covmat = np.dot(centered_images.T, centered_images) / N  ## 1/N (x-u)^t (x-u)
        covmat += reg * np.eye(covmat.shape[0])
        params[c] = (mean, covmat)
    return params
    

In [ ]:
def LDA(test_images, test_labels, params):
    pass

In [ ]:
def QDA():
    pass

In [2]:
def visualise(x_train, y_train, x_test, y_test):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    print("Computing t-SNE for Train set...")
    tsne_train = TSNE(n_components=2, random_state=42, perplexity=30)
    X_train_embedded = tsne_train.fit_transform(x_train)
    
    scatter1 = axes[0].scatter(X_train_embedded[:, 0], X_train_embedded[:, 1], c=y_train, cmap='viridis', alpha=0.7)
    axes[0].set_title("t-SNE - Train Set")
    axes[0].legend(*scatter1.legend_elements(), title="Classes")
    
    # 2. Test Set t-SNE
    print("Computing t-SNE for Test set...")
    tsne_test = TSNE(n_components=2, random_state=42, perplexity=30)
    X_test_embedded = tsne_test.fit_transform(x_test)
    
    scatter2 = axes[1].scatter(X_test_embedded[:, 0], X_test_embedded[:, 1], c=y_test, cmap='viridis', alpha=0.7)
    axes[1].set_title("t-SNE - Test Set")
    axes[1].legend(*scatter2.legend_elements(), title="Classes")
    
    plt.show()
    pass

In [ ]:
train_images, train_labels = load_dataset(train_images_fpath, train_labels_fpath)
test_images, test_labels = load_dataset(test_images_fpath, test_labels_fpath)
